# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My lane's question is "which pages should we prioritize for refresh" ; a ranking problem over an observed yes/no outcome (declined_next_month). Per the toolkit table, that shape calls for Logistic Regression first (readable), then Random Forest (stronger); both produce probability scores I can also evaluate at Precision@K, since ranking needs scores, not just labels. I'm not reaching for Gradient Boosting or clustering: my lane isn't a grouping problem, and added complexity should only show up if it earns its keep over the simpler models.

In [1]:
%pip -q install duckdb scikit-learn
import os, getpass
import duckdb
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
FEATURE_MONTH = '2026-03'
OUTCOME_MONTH = '2026-04'

monthly = con.sql(f"""
    SELECT client_hash_id, content_hash_id,
        SUM(CASE WHEN month = '{FEATURE_MONTH}' THEN gsc_impressions ELSE 0 END) AS imp_march,
        SUM(CASE WHEN month = '{FEATURE_MONTH}' THEN gsc_clicks ELSE 0 END) AS clk_march,
        AVG(CASE WHEN month = '{FEATURE_MONTH}' AND gsc_avg_position > 0 THEN gsc_avg_position END) AS pos_march,
        COUNT(*) FILTER (WHERE month = '{FEATURE_MONTH}' AND gsc_impressions > 0) AS active_days_march,
        SUM(CASE WHEN month = '{OUTCOME_MONTH}' THEN gsc_impressions ELSE 0 END) AS imp_april
    FROM {FACT}
    WHERE month IN ('{FEATURE_MONTH}', '{OUTCOME_MONTH}')
    GROUP BY 1,2
    HAVING imp_march >= 100
""").df()

monthly['ctr_march'] = (monthly['clk_march'] / monthly['imp_march']).round(4)
monthly['declined_next_month'] = (monthly['imp_april'] < 0.8 * monthly['imp_march']).astype(int)

content = con.sql(f"""
    SELECT content_hash_id,
        DATE_DIFF('day', content_updated_date, DATE '{FEATURE_MONTH}-01') AS days_since_last_update
    FROM {DIM_CONTENT}
""").df()

df = monthly.merge(content, on='content_hash_id', how='left')
df.loc[df['days_since_last_update'] < 0, 'days_since_last_update'] = pd.NA

# honest missingness handling: a flag + a filled numeric, not a silent guess
df['staleness_unknown'] = df['days_since_last_update'].isna().astype(int)
df['days_since_last_update_filled'] = df['days_since_last_update'].fillna(-1)

print(f"{len(df):,} rows ready for modeling")
df.head()

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

101,441 rows ready for modeling


,client_hash_id,content_hash_id,imp_march,clk_march,pos_march,active_days_march,imp_april,ctr_march,declined_next_month,days_since_last_update,staleness_unknown,days_since_last_update_filled
0,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,602.0,4.0,4.428747,29,879.0,0.0066,0,NaN,1,-1.0
1,client_62f4a7e64f5e0096,content_275b6f7f733016d4,810.0,1.0,4.866123,29,335.0,0.0012,1,NaN,1,-1.0
2,client_62f4a7e64f5e0096,content_755d951187fcd70a,1858.0,6.0,1.854929,30,1587.0,0.0032,0,NaN,1,-1.0
3,client_62f4a7e64f5e0096,content_92c381fbd361212e,536.0,1.0,4.442543,29,242.0,0.0019,1,NaN,1,-1.0
4,client_62f4a7e64f5e0096,content_97188a7032a705cf,496.0,3.0,4.018509,29,346.0,0.0060,1,NaN,1,-1.0


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I'm using a grouped split by client_hash_id, not a random row split. Content items are nested within clients, and a client's pages share traffic scale, industry, and update habits; a random split would let the model see other pages from the same client in training and leak client-level patterns into the test score. GroupShuffleSplit keeps every client entirely in train or entirely in test, so the test score reflects generalizing to unseen clients, which is the honest version of this question.

In [2]:
FEATURES = ['imp_march', 'ctr_march', 'pos_march', 'active_days_march',
            'days_since_last_update_filled', 'staleness_unknown']

model_df = df.dropna(subset=['pos_march']).copy()
X = model_df[FEATURES]
y = model_df['declined_next_month']
groups = model_df['client_hash_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
test_df = model_df.iloc[test_idx].copy()

print(f"Train: {len(X_train):,} rows, {model_df.iloc[train_idx]['client_hash_id'].nunique()} clients")
print(f"Test:  {len(X_test):,} rows, {test_df['client_hash_id'].nunique()} clients")
print(f"Base rate (declined) — train: {y_train.mean():.3f}, test: {y_test.mean():.3f}")

Train: 93,085 rows, 33 clients
Test:  8,356 rows, 11 clients
Base rate (declined) — train: 0.520, test: 0.484


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
def precision_at_k(y_true, scores, k):
    order = np.argsort(-scores)[:k]
    return y_true.values[order].mean()

#baseline rule, recomputed on the TEST split only
test_df['stale'] = (test_df['days_since_last_update'] >= 180).fillna(False).astype(int)
test_df['visible'] = (test_df['imp_march'] >= 500).astype(int)
test_df['baseline_score'] = test_df['stale'] * test_df['visible'] * test_df['imp_march']

#Logistic Regression (scaled)
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)
X_train_scaled = scaler.transform(X_train)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=1000, random_state=42).fit(X_train_scaled, y_train)
logreg_scores = logreg.predict_proba(X_test_scaled)[:, 1]

# Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42).fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]

K = 50
results = pd.DataFrame({
    'method': ['Baseline rule', 'Logistic Regression', 'Random Forest'],
    'AUC': [
        roc_auc_score(y_test, test_df['baseline_score']),
        roc_auc_score(y_test, logreg_scores),
        roc_auc_score(y_test, rf_scores),
    ],
    f'Precision@{K}': [
        precision_at_k(y_test, test_df['baseline_score'].values, K),
        precision_at_k(y_test, logreg_scores, K),
        precision_at_k(y_test, rf_scores, K),
    ],
    'base_rate': [y_test.mean()] * 3,
}).round(3)
results

,method,AUC,Precision@50,base_rate
0,Baseline rule,0.500,0.56,0.484
1,Logistic Regression,0.648,0.40,0.484
2,Random Forest,0.672,0.72,0.484


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [6]:
# feature importance; sanity check for leakage (suspiciously perfect = red flag)
coef_table = pd.Series(logreg.coef_[0], index=FEATURES).sort_values(key=abs, ascending=False)
importance_table = pd.Series(rf.feature_importances_, index=FEATURES).sort_values(ascending=False)
print("Logistic Regression coefficients:\n", coef_table, "\n")
print("Random Forest importances:\n", importance_table)

# where is the model most wrong?
test_df['rf_score'] = rf_scores
test_df['rf_pred'] = (rf_scores >= 0.5).astype(int)
errors = test_df[test_df['rf_pred'] != test_df['declined_next_month']]
print(f"\n{len(errors):,} / {len(test_df):,} test rows misclassified")

# 3 concrete wrong cases
errors.sort_values('rf_score', ascending=False)[
    ['content_hash_id', 'imp_march', 'ctr_march', 'pos_march', 'rf_score', 'declined_next_month']
].head(3)

Logistic Regression coefficients:
 ctr_march                       -0.410584
active_days_march                0.343538
imp_march                       -0.138366
pos_march                       -0.104029
staleness_unknown               -0.097390
days_since_last_update_filled    0.019137
dtype: float64 

Random Forest importances:
 ctr_march                        0.371189
active_days_march                0.336509
imp_march                        0.122994
pos_march                        0.094654
days_since_last_update_filled    0.042294
staleness_unknown                0.032359
dtype: float64

3,099 / 8,356 test rows misclassified


,content_hash_id,imp_march,ctr_march,pos_march,rf_score,declined_next_month
17547,content_246fdbfcd88a4611,229.0,0.0,6.935652,0.714253,0
17528,content_b2bb1a2f3763c8bd,129.0,0.0,5.396480,0.711267,0
68262,content_2d782d6b44224dc7,145.0,0.0,9.145816,0.702348,0


Top features: Random Forest leans most on ctr_march (0.371) and active_days_march (0.337), with imp_march (0.123) and pos_march (0.095) a clear second tier — no single feature dominates to a suspicious degree, and the pattern makes sense: a page's own click-through behavior and how consistently it showed impressions across March are reasonable early signals of a coming drop.
The baseline rule's AUC of 0.500 isn't a real tie with chance - it's an artifact of scoring most rows 0 under a hard threshold rule, which collapses ROC ordering among ties; Precision@50 (0.56) is the fairer read of the rule's actual signal.
After rescaling, Logistic Regression's coefficients are all in a comparable, sane range (-0.41 to +0.34) with no single feature dominating - ruling out the earlier number as a leakage signal and confirming it was purely a scaling artifact. Even properly scaled, though, Logistic Regression's Precision@50 (0.40) still comes in below the base rate (0.484) despite a respectable overall AUC (0.648) ; a genuine finding, not a bug: it discriminates reasonably well across the whole range but its linear decision boundary doesn't capture the extreme, most-confident end of the ranking as well as Random Forest does. The model is most wrong on low-volume, zero-CTR, poorly-ranked pages (all three example errors have ctr_march = 0.0, position 5-9, and under 250 March impressions) that Random Forest confidently flags as declining (score ~0.70) but that actually held steady - likely a floor effect: pages already this weak have little room left to fall 20% further, so the 0.8x threshold is easy to avoid even without real recovery. Worth noting: the test split has only 11 clients, so these numbers could shift with a different random seed, a real limitation of grouping by client on a dataset this size.

## Self-check

Before you submit, confirm each line honestly:

- [✅] Every section above is filled — markdown thinking AND the code that backs it
- [✅] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅] No client names, URLs, or private queries anywhere
- [✅] My claims use careful words: observed, measured, directional, decision-support
- [✅] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.